# Conversational Memory Bot with LangGraph

### Objectives:
1. **Use Different Message Types**: Handle `HumanMessage`, `AIMessage`, and `SystemMessage` explicitly.
2. **Maintain Full Conversation History**: Persist multi-turn dialogs across invocations using state message reducers (`add_messages`) and checkpoint memory (`MemorySaver`).
3. **Use GPT-4o with ChatOpenAI**: Initialize and configure the `gpt-4o` model via `langchain-openai`.
4. **Create a Sophisticated Conversation Loop**: An interactive multi-turn loop supporting conversational commands (e.g., `exit`, `history`, `clear`), thread IDs, and memory persistence.

**Main Goal:** Implement persistent conversation memory so the agent remembers facts and context across dialogue turns.

## 0. Dependencies and Imports

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core langchain-openai python-dotenv

In [ ]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv

# LangChain Message Types
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langchain_openai import ChatOpenAI

# LangGraph State & Checkpoint Memory
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Load environment variables
load_dotenv()

## 1. Objective 1 & 2: Define State Structure with Full History Preservation

By annotating `messages` with `add_messages`, LangGraph automatically preserves the entire history of `HumanMessage` and `AIMessage` objects without overriding past turns.

In [ ]:
class ConversationState(TypedDict):
    """
    State containing full conversation history of HumanMessage and AIMessage objects.
    The add_messages reducer appends new messages into the ongoing history.
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]

## 2. Objective 3: Initialize GPT-4o Model

We instantiate `ChatOpenAI(model="gpt-4o")`.

In [ ]:
# Initialize GPT-4o
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)

print("Initialized model:", llm.model_name)

## 3. Node Function: Generating AI Responses

In [ ]:
# System prompt defining the persona and behavior
SYSTEM_PROMPT = SystemMessage(
    content="You are an intelligent, empathetic conversational AI assistant. You remember all past facts shared by the user in the conversation and refer to them naturally."
)

def chatbot_node(state: ConversationState) -> dict:
    """
    Node that takes the complete message history, ensures a system message is included,
    invokes GPT-4o, and returns an AIMessage.
    """
    current_messages = list(state["messages"])
    
    # Prepend SystemMessage if not already present
    if not current_messages or not isinstance(current_messages[0], SystemMessage):
        conversation_with_system = [SYSTEM_PROMPT] + current_messages
    else:
        conversation_with_system = current_messages
        
    # Invoke LLM
    ai_response = llm.invoke(conversation_with_system)
    
    # Returning AIMessage will append to state.messages via add_messages reducer
    return {"messages": [ai_response]}

## 4. Main Goal: Checkpointer Memory (`MemorySaver`)

By passing a `checkpointer=MemorySaver()` during graph compilation, LangGraph persists the conversation state per `thread_id`.
- Multiple users or sessions can have distinct thread IDs.
- In subsequent calls with the same `thread_id`, the agent **retains full memory** of previous turns without needing to resend the entire message history manually!

In [ ]:
# Create Checkpointer for persistent in-memory session retention
checkpointer = MemorySaver()

# Build Graph
builder = StateGraph(ConversationState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Compile Graph with Checkpointer Memory
agent_with_memory = builder.compile(checkpointer=checkpointer)
print("Agent compiled with persistent MemorySaver checkpointing!")

## 5. Simulating Multi-Turn Memory Verification

Let's simulate two consecutive turns on `thread_id: "session_123"`:
1. **Turn 1**: User shares a secret or personal detail (`HumanMessage`: *"My favorite fruit is Mango and I live in Paris."*)
2. **Turn 2**: User asks a recall question (`HumanMessage`: *"What fruit do I like and where do I live?"*)

In [ ]:
# Session configuration thread
config = {"configurable": {"thread_id": "session_123"}}

print("=== TURN 1 ===")
turn1_input = {"messages": [HumanMessage(content="Hi! My favorite fruit is Mango and I live in Paris.")]}
try:
    turn1_output = agent_with_memory.invoke(turn1_input, config=config)
    print("AI:", turn1_output["messages"][-1].content)
except Exception as e:
    print(f"(API call note: {e})")

print("\n=== TURN 2 (Testing Memory Recall) ===")
# Note: We only pass the NEW HumanMessage; LangGraph retrieves past history from MemorySaver
turn2_input = {"messages": [HumanMessage(content="What fruit do I like, and where do I live?")]}
try:
    turn2_output = agent_with_memory.invoke(turn2_input, config=config)
    print("AI:", turn2_output["messages"][-1].content)
    
    print("\n--- Complete Stored Message History in Thread ---")
    for i, msg in enumerate(turn2_output["messages"]):
        print(f"{i+1}. [{msg.__class__.__name__}]: {msg.content}")
except Exception as e:
    print(f"(API call note: {e})")

## 6. Objective 4: Sophisticated Interactive Conversation Loop

A full interactive loop that:
- Accepts continuous user input (`HumanMessage`)
- Generates AI replies (`AIMessage`)
- Supports special commands: `/history` (view turns), `/clear` (reset memory), `/exit` (stop loop)
- Maintains persistent conversational memory across turns

In [ ]:
import uuid

def run_interactive_conversation_loop(agent, thread_id: str = None):
    """
    Runs an interactive multi-turn conversation loop with memory persistence.
    """
    if thread_id is None:
        thread_id = str(uuid.uuid4())[:8]
        
    config = {"configurable": {"thread_id": thread_id}}
    print(f"=======================================================")
    print(f"  Starting Conversation Loop (Thread ID: {thread_id})")
    print(f"  Commands: 'exit' to quit | 'history' to view log | 'clear' to reset")
    print(f"=======================================================\n")
    
    while True:
        try:
            user_text = input("User: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting session.")
            break
            
        if not user_text:
            continue
            
        if user_text.lower() in ["exit", "quit", "q"]:
            print("Session ended. Goodbye!")
            break
            
        elif user_text.lower() == "history":
            snapshot = agent.get_state(config)
            messages = snapshot.values.get("messages", [])
            print(f"\n--- Current Conversation History ({len(messages)} messages) ---")
            for idx, msg in enumerate(messages):
                prefix = "User (HumanMessage)" if isinstance(msg, HumanMessage) else "AI (AIMessage)"
                print(f"{idx + 1}. {prefix}: {msg.content}")
            print("-----------------------------------------------------------\n")
            continue
            
        elif user_text.lower() == "clear":
            # Start a new thread for clean memory
            thread_id = str(uuid.uuid4())[:8]
            config = {"configurable": {"thread_id": thread_id}}
            print(f"Memory cleared! Started new thread: {thread_id}\n")
            continue
            
        # Create HumanMessage and invoke agent
        human_msg = HumanMessage(content=user_text)
        try:
            response_state = agent.invoke({"messages": [human_msg]}, config=config)
            ai_msg = response_state["messages"][-1]
            print(f"AI: {ai_msg.content}\n")
        except Exception as e:
            print(f"[Error invoking agent: {e}]\n")

# To run the interactive loop in your local environment, uncomment:
# run_interactive_conversation_loop(agent_with_memory)